# Annual single-pass pairwise agreement statistics

Pool exact sufficient sums and mergeable distribution summaries from the v5 daily cache. No full-resolution ERA5 results are read by this notebook.

In [ ]:
from datetime import date
from pathlib import Path

import json
import numpy as np
import pandas as pd
import pyarrow.parquet as pq
import plotly.graph_objects as go
from IPython.display import display

from liljegren_analysis import histogram_quantile_statistics, pooled_sufficient_statistics


In [ ]:
repository = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
pairwise_root = repository / "data" / "pairwise" / "all-implementations" / "v1"
year = 2025
start_date = date(year, 1, 1)
end_date = date(year, 12, 31)


In [ ]:
metadata = pd.DataFrame([json.loads((pairwise_root / "metadata.json").read_text())])
date_directories = [
    pairwise_root / "dates" / (start_date.fromordinal(start_date.toordinal() + offset)).isoformat()
    for offset in range((end_date - start_date).days + 1)
]
def load(name):
    return pd.concat([pq.read_table(directory / f"{name}.parquet").to_pandas() for directory in date_directories])

daily = load("pairwise")
histogram = load("abs_histogram").groupby(
    ["quantity", "implementation_a", "implementation_b", "abs_bin_id"], as_index=False, sort=True
).agg(abs_bin_lower_c=("abs_bin_lower_c", "min"), abs_bin_upper_c=("abs_bin_upper_c", "min"), n=("n", "sum"), area_weight_sum=("area_weight_sum", "sum"))
quantity_status = load("quantity_status")


In [ ]:
if metadata.to_dict("records") != [{
    "schema_version": 5,
    "sample_scope": "all-implementations",
    "result_schema_id": "hsb-wbgt-comparison-results-csv-v3",
    "quantity_scope": "joint",
    "quantity_set_version": 1,
    "implementation_set_version": 1,
    "abs_histogram_version": 1,
}]:
    raise ValueError(f"Unexpected pairwise schema metadata:\n{metadata}")
quantities = ("wbgt", "tg", "tnwb")
pair_columns = ["quantity", "implementation_a", "implementation_b"]
expected_dates = (end_date - start_date).days + 1
coverage = daily.groupby(pair_columns)["result_date"].nunique()
if len(coverage) != 63 or not coverage.eq(expected_dates).all():
    raise ValueError(f"Incomplete daily pair coverage:\n{coverage}")

joint_support = daily.groupby(
    ["result_date", "implementation_a", "implementation_b"]
)[["n", "area_weight_sum"]].nunique()
if joint_support.gt(1).any().any():
    raise ValueError(f"Joint quantity populations differ:\n{joint_support}")

quantity_status["valid"] = (
    quantity_status["status"].eq("ok") & quantity_status["value_present"]
)
quantity_coverage = (
    quantity_status
    .assign(valid_n=lambda frame: frame["n"].where(frame["valid"], 0))
    .groupby(["quantity", "implementation"], as_index=False, sort=True)
    .agg(total_n=("n", "sum"), valid_n=("valid_n", "sum"))
)
quantity_coverage["valid_percent"] = (
    100.0 * quantity_coverage["valid_n"] / quantity_coverage["total_n"]
)
display(coverage)
display(quantity_coverage)

In [ ]:
records = []
for pair_key, pair in daily.groupby(pair_columns, sort=True):
    quantity, implementation_a, implementation_b = pair_key
    statistics = pooled_sufficient_statistics(
        pair["n"],
        sum_differences=pair["sum_difference_c"],
        sum_squared_differences=pair["sum_squared_difference_c2"],
        sum_absolute_differences=pair["sum_absolute_difference_c"],
        min_differences=pair["min_difference_c"],
        max_differences=pair["max_difference_c"],
        max_absolute_differences=pair["max_absolute_difference_c"],
        area_weight_sums=pair["area_weight_sum"],
        area_weighted_sum_differences=pair["area_weighted_difference_sum_c"],
        area_weighted_sum_squared_differences=pair["area_weighted_squared_difference_sum_c2"],
        area_weighted_sum_absolute_differences=pair["area_weighted_absolute_difference_sum_c"],
        exceedance_counts_0_1=pair["n_absolute_difference_gt_0_1_c"],
        exceedance_counts_0_2=pair["n_absolute_difference_gt_0_2_c"],
        exceedance_counts_0_5=pair["n_absolute_difference_gt_0_5_c"],
        exceedance_area_weights_0_1=pair["area_weight_absolute_difference_gt_0_1_c"],
        exceedance_area_weights_0_2=pair["area_weight_absolute_difference_gt_0_2_c"],
        exceedance_area_weights_0_5=pair["area_weight_absolute_difference_gt_0_5_c"],
    )
    pair_histogram = histogram.loc[
        histogram["quantity"].eq(quantity)
        & histogram["implementation_a"].eq(implementation_a)
        & histogram["implementation_b"].eq(implementation_b)
    ]
    quantiles = histogram_quantile_statistics(
        pair_histogram["abs_bin_id"],
        bin_lowers=pair_histogram["abs_bin_lower_c"],
        bin_uppers=pair_histogram["abs_bin_upper_c"],
        counts=pair_histogram["n"],
        area_weight_sums=pair_histogram["area_weight_sum"],
        expected_count=statistics["n"],
        expected_area_weight_sum=statistics["area_weight_sum"],
    )
    records.append({
        "quantity": quantity,
        "implementation_a": implementation_a,
        "implementation_b": implementation_b,
        **statistics,
        **quantiles,
    })

yearly_pairwise = pd.DataFrame.from_records(records)

yearly_pairwise

## Annual agreement matrices

In [ ]:
statistic_specs = {
    "n": {"label": "Matched observations", "unit": "rows"},
    "mean_difference_c": {"label": "Mean difference", "unit": "°C", "signed": True, "reverse_sign": True},
    "stddev_difference_c": {"label": "Difference SD", "unit": "°C"},
    "mean_absolute_difference_c": {"label": "MAE", "unit": "°C"},
    "rmse_c": {"label": "RMSE", "unit": "°C"},
    "loa_lower_c": {"label": "Lower parametric LoA", "unit": "°C", "signed": True, "reverse_sign": True, "reverse_statistic": "loa_upper_c"},
    "loa_upper_c": {"label": "Upper parametric LoA", "unit": "°C", "signed": True, "reverse_sign": True, "reverse_statistic": "loa_lower_c"},
    "median_absolute_difference_c": {"label": "Approximate P50 |difference|", "unit": "°C"},
    "p95_absolute_difference_c": {"label": "Approximate P95 |difference|", "unit": "°C"},
    "p99_absolute_difference_c": {"label": "Approximate P99 |difference|", "unit": "°C"},
    "max_absolute_difference_c": {"label": "Maximum |difference|", "unit": "°C"},
    "area_weighted_mean_difference_c": {"label": "Area-weighted mean difference", "unit": "°C", "signed": True, "reverse_sign": True},
    "area_weighted_stddev_difference_c": {"label": "Area-weighted difference SD", "unit": "°C"},
    "area_weighted_mean_absolute_difference_c": {"label": "Area-weighted MAE", "unit": "°C"},
    "area_weighted_rmse_c": {"label": "Area-weighted RMSE", "unit": "°C"},
    "area_weighted_loa_lower_c": {"label": "Area-weighted lower parametric LoA", "unit": "°C", "signed": True, "reverse_sign": True, "reverse_statistic": "area_weighted_loa_upper_c"},
    "area_weighted_loa_upper_c": {"label": "Area-weighted upper parametric LoA", "unit": "°C", "signed": True, "reverse_sign": True, "reverse_statistic": "area_weighted_loa_lower_c"},
    "area_weighted_median_absolute_difference_c": {"label": "Area-weighted approximate P50 |difference|", "unit": "°C"},
    "area_weighted_p95_absolute_difference_c": {"label": "Area-weighted approximate P95 |difference|", "unit": "°C"},
    "area_weighted_p99_absolute_difference_c": {"label": "Area-weighted approximate P99 |difference|", "unit": "°C"},
}

implementations = sorted(set(yearly_pairwise["implementation_a"]) | set(yearly_pairwise["implementation_b"]))
quantity_matrices = {}

for quantity in quantities:
    statistic_matrices = {
        statistic: pd.DataFrame(np.nan, index=implementations, columns=implementations)
        for statistic in statistic_specs
    }
    for row in yearly_pairwise.loc[yearly_pairwise["quantity"].eq(quantity)].itertuples(index=False):
        for statistic, spec in statistic_specs.items():
            value = getattr(row, statistic)
            statistic_matrices[statistic].loc[row.implementation_a, row.implementation_b] = value
            reverse_value = getattr(row, spec.get("reverse_statistic", statistic))
            statistic_matrices[statistic].loc[row.implementation_b, row.implementation_a] = -reverse_value if spec.get("reverse_sign") else reverse_value
    quantity_matrices[quantity] = statistic_matrices

In [ ]:
quantity_labels = {"wbgt": "WBGT", "tg": "Tg", "tnwb": "Tnwb"}
figures = {}
statistics = list(statistic_specs)
default_statistic = "mean_difference_c"
default_position = statistics.index(default_statistic)

for quantity in quantities:
    statistic_matrices = quantity_matrices[quantity]
    statistic_domains = {}
    for statistic in statistics:
        counterpart = statistic.removeprefix("area_weighted_") if statistic.startswith("area_weighted_") else f"area_weighted_{statistic}"
        domain_statistics = (statistic, counterpart) if counterpart in statistic_matrices else (statistic,)
        finite = np.concatenate([
            values[np.isfinite(values)]
            for domain_statistic in domain_statistics
            for values in [statistic_matrices[domain_statistic].to_numpy()]
        ])
        minimum, maximum = float(np.min(finite)), float(np.max(finite))
        if minimum == maximum:
            padding = max(abs(minimum) * 1e-12, np.finfo(float).eps)
            minimum, maximum = minimum - padding, maximum + padding
        statistic_domains[statistic] = (minimum, maximum)

    figure = go.Figure()
    for position, statistic in enumerate(statistics):
        spec = statistic_specs[statistic]
        matrix = statistic_matrices[statistic]
        minimum, maximum = statistic_domains[statistic]
        signed = spec.get("signed", False)
        figure.add_trace(go.Heatmap(
            z=matrix.to_numpy(), x=implementations, y=implementations,
            text=matrix.map(lambda value: "" if pd.isna(value) else f"{value:,.2f}").to_numpy(),
            texttemplate="%{text}", colorscale="RdBu_r" if signed else "YlOrRd",
            zmin=minimum, zmax=maximum, zmid=0 if signed else None,
            colorbar={"title": {"text": spec["unit"]}, "tickformat": ",.2f"}, visible=position == default_position,
            hovertemplate="<b>%{y} − %{x}</b><br>" + spec["label"] + ": %{z:,.2f} " + spec["unit"] + "<extra></extra>",
        ))

    figure.update_layout(
        template="plotly_dark", title=f"{quantity_labels[quantity]}",
        height=760, margin={"l": 120, "r": 80, "t": 180, "b": 100},
        updatemenus=[{
            "buttons": [{"label": statistic_specs[s]["label"], "method": "update", "args": [{"visible": [candidate == s for candidate in statistics]}]} for s in statistics],
            "direction": "down", "showactive": True, "active": default_position,
            "x": 0, "xanchor": "left", "y": 1.08, "yanchor": "bottom",
        }],
        annotations=[{"text": "Statistic", "showarrow": False, "x": 0, "xref": "paper", "xanchor": "left", "y": 1.18, "yref": "paper", "yanchor": "bottom"}],
        xaxis={"title": "implementation_b"},
        yaxis={"title": "implementation_a", "autorange": "reversed", "scaleanchor": "x", "scaleratio": 1},
    )
    figures[quantity] = figure
    display(figure)

In [ ]:
quantity_matrices['wbgt']['mean_difference_c']['liljegren-c']

In [ ]:
def pairwise_metric_vs_reference(metric, reference, retained_rows):
    comparison = pd.DataFrame({
        quantity_labels[quantity]: quantity_matrices[quantity][metric][reference]
        for quantity in ('wbgt', 'tnwb', 'tg')
    })
    return comparison.loc[list(retained_rows)].round(2)


In [ ]:
pairwise_metric_vs_reference(
    'mean_difference_c',
    'liljegren-c',
    ('heatstress', 'meteor', 'pywbgt', 'thermofeel'),
)

In [ ]:
def metric_sheet_name(metric):
    name = (
        metric
        .replace('area_weighted_', 'aw_')
        .replace('absolute', 'abs')
        .replace('difference', 'diff')
    )
    if len(name) > 31:
        raise ValueError(f'Excel worksheet name is too long: {name}')
    return name

def export_pairwise_metrics_to_xlsx(metrics, reference, retained_rows, output_path):
    metrics = tuple(metrics)
    if not metrics:
        raise ValueError('At least one metric is required')
    output_path = Path(output_path)
    output_path.parent.mkdir(parents=True, exist_ok=True)
    sheet_names = [metric_sheet_name(metric) for metric in metrics]
    if len(sheet_names) != len(set(sheet_names)):
        raise ValueError('Metric worksheet names are not unique')

    with pd.ExcelWriter(output_path, engine='openpyxl') as writer:
        for metric, sheet_name in zip(metrics, sheet_names, strict=True):
            comparison = pairwise_metric_vs_reference(metric, reference, retained_rows)
            comparison.rename_axis('implementation').to_excel(writer, sheet_name=sheet_name)
            worksheet = writer.sheets[sheet_name]
            worksheet.freeze_panes = 'B2'
            worksheet.column_dimensions['A'].width = max(16, max(map(len, retained_rows)) + 2)
            for column in ('B', 'C', 'D'):
                worksheet.column_dimensions[column].width = 14
            for row in worksheet.iter_rows(
                min_row=2, max_row=len(comparison) + 1, min_col=2, max_col=4,
            ):
                for cell in row:
                    cell.number_format = '#,##0' if metric == 'n' else '0.00'
    return output_path

export_metrics = (
    'n',
    *(metric for metric in statistic_specs if metric.startswith('area_weighted_')),
)
xlsx_path = export_pairwise_metrics_to_xlsx(
    metrics=export_metrics,
    reference='liljegren-c',
    retained_rows=('heatstress', 'meteor', 'thermofeel','pywbgt'),
    output_path=repository / 'data' / 'pairwise_metrics_vs_liljegren-c.xlsx',
)
xlsx_path
